# RAFT++: Rejection Sampling Fine-Tuning with GVM Dynamic Allocation

This notebook applies **RAFT++** (Rejection Sampling Fine-Tuning) with **GVM-RAFT**
dynamic completion allocation to improve STEM reasoning.

**Training pipeline stage:** 3 of 5 (SFT -> GSPO (curriculum) -> **RAFT++** -> AdaSTaR -> DPO)

**Target hardware:** Google Colab A100 40GB / 80GB

**Key features:**
- **GVM-RAFT dynamic allocation (arXiv 2504.11343):** Pilot round (4 completions) estimates
  per-problem difficulty, remaining budget allocated proportionally — hard problems get more attempts.
  2-4x speedup over fixed allocation.
- **Negative saving (arXiv 2505.24850):** Incorrect completions saved to JSONL for DPO stage reuse,
  eliminating redundant generation in the final pipeline stage.
- Domain-specific verification with verify_answers.py
- Domain balancing: oversample minority domains to prevent imbalance
- Up to 2 RAFT rounds with early stopping
- Fresh LoRA adapter on top of GSPO checkpoint
- No negative penalties: only learn from correct solutions

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

**References:**
- [GVM-RAFT / Minimalist Approach (arXiv 2504.11343)](https://arxiv.org/abs/2504.11343) — Dynamic allocation
- [Harnessing Negative Signals (arXiv 2505.24850)](https://arxiv.org/abs/2505.24850) — Reusing incorrect completions
- [ReST (arXiv 2308.08998)](https://arxiv.org/abs/2308.08998) — Reinforced Self-Training
- [STaR (arXiv 2203.14465)](https://arxiv.org/abs/2203.14465) — Self-Taught Reasoner

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# Install dependencies — TRL >= 0.27.0 pinned for consistency across pipeline
!pip install -q unsloth "trl>=0.27.0" peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verifiable reward functions

# Login to HuggingFace
from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Add Drive root to sys.path (training/ is at MyDrive level)
# ============================================================
import sys, os

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model — Instruct base gives dialogue abilities built-in
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
GSPO_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3_4b/final_adapter"
GSPO_HF_REPO = "Siesher/mits-qwen3-4b-gspo"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/raft_qwen3_4b"

# ---- A100 GPU preset ----
A100_VRAM_GB = 40

# RAFT++ parameters
if A100_VRAM_GB >= 80:
    TOTAL_BUDGET_PER_PROBLEM = 32   # Total completions budget per problem
    SFT_BATCH_SIZE = 4
else:
    TOTAL_BUDGET_PER_PROBLEM = 16   # A100 40GB
    SFT_BATCH_SIZE = 2

MAX_ROUNDS = 2                 # RAFT++ rounds
EARLY_STOP_THRESHOLD = 0.01   # Stop if accuracy improvement < 1%
MAX_COMPLETION = 1024
MAX_PROMPT_LENGTH = 512
GENERATION_TEMPERATURE = 0.9  # Diverse completions

# ---- GVM-RAFT: Dynamic completion allocation (arXiv 2504.11343) ----
# Instead of fixed N completions per problem, pilot round estimates difficulty,
# then remaining budget is allocated proportionally — hard problems get more attempts.
GVM_PILOT_SIZE = 4             # Completions in pilot round per problem
GVM_MIN_ADDITIONAL = 2         # Minimum additional completions for unsolved problems

# ---- RAFT++ Negative Saving (arXiv 2505.24850) ----
# Save incorrect completions for DPO stage reuse
NEGATIVES_PATH = os.path.join(OUTPUT_DIR, "raft_negatives.jsonl") if 'OUTPUT_DIR' in dir() else ""

# SFT parameters for RAFT
SFT_LR = 2e-5
SFT_EPOCHS = 1                # 1 epoch per RAFT round
SFT_GRAD_ACCUM = 4
SFT_MAX_SEQ_LENGTH = MAX_PROMPT_LENGTH + MAX_COMPLETION

# LoRA for RAFT SFT
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Domains
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

SYSTEM_PROMPT = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."

print(f"Hardware: A100 {A100_VRAM_GB}GB")
print(f"Base model: {BASE_MODEL}")
print(f"RAFT++: budget={TOTAL_BUDGET_PER_PROBLEM}/problem, {MAX_ROUNDS} rounds")
print(f"GVM-RAFT: pilot={GVM_PILOT_SIZE}, dynamic allocation enabled")
print(f"Negative saving: enabled (for DPO reuse)")
print(f"Early stop threshold: {EARLY_STOP_THRESHOLD}")
print(f"SFT: lr={SFT_LR}, epochs={SFT_EPOCHS}, batch={SFT_BATCH_SIZE}")

In [ ]:
# ============================================================
# Mount Drive + resolve GSPO checkpoint
# ============================================================
import json
import os

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/raft_qwen3_4b"

# Resolve GSPO checkpoint: Drive first, then download from HuggingFace
if os.path.exists(GSPO_CHECKPOINT):
    print(f"GSPO checkpoint found on Drive: {GSPO_CHECKPOINT}")
elif GSPO_HF_REPO:
    from huggingface_hub import snapshot_download
    GSPO_CHECKPOINT = snapshot_download(GSPO_HF_REPO)
    print(f"Downloaded GSPO adapter from HuggingFace to: {GSPO_CHECKPOINT}")
else:
    raise FileNotFoundError(f"GSPO checkpoint not found: {GSPO_CHECKPOINT}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Load RL problems: Drive JSONL → HF "rl" config → HF "gspo" fallback
# ============================================================
import json
from datasets import load_dataset
from collections import Counter, defaultdict

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

def load_rl_problems():
    """Load RL dataset with fallback chain."""
    if os.path.exists(RL_DATA_PATH):
        print(f"Loading RL data from Drive: {RL_DATA_PATH}")
        problems = []
        with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
            for line in f:
                problems.append(json.loads(line))
        print(f"Loaded {len(problems)} problems from Drive JSONL")
        return problems
    try:
        print("Trying HF dataset config 'rl'...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'rl' config")
        return problems
    except Exception:
        pass
    print("Falling back to HF 'gspo' config...")
    hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
    problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
    print(f"Loaded {len(problems)} from HF 'gspo' config (fallback)")
    return problems


problems = load_rl_problems()
verifiable_problems = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]

print(f"Loaded {len(verifiable_problems)} verifiable problems")
domain_counts = Counter(p.get("domain", "unknown") for p in verifiable_problems)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count}")

In [ ]:
# ============================================================
# Load model + GSPO adapter
# ============================================================
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=SFT_MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

# Load GSPO adapter config
gspo_config_path = os.path.join(GSPO_CHECKPOINT, "adapter_config.json")
if os.path.exists(gspo_config_path):
    with open(gspo_config_path) as f:
        gspo_cfg = json.load(f)
    gspo_r = gspo_cfg.get("r", LORA_R)
    gspo_alpha = gspo_cfg.get("lora_alpha", LORA_ALPHA)
    print(f"GSPO adapter config: r={gspo_r}, alpha={gspo_alpha}")
else:
    gspo_r = LORA_R
    gspo_alpha = LORA_ALPHA

effective_r = max(gspo_r, LORA_R)

# Create fresh LoRA for RAFT SFT
model = FastLanguageModel.get_peft_model(
    model,
    r=effective_r,
    lora_alpha=gspo_alpha,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Load GSPO adapter weights
from safetensors.torch import load_file
gspo_weights_path = os.path.join(GSPO_CHECKPOINT, "adapter_model.safetensors")
if os.path.exists(gspo_weights_path):
    gspo_weights = load_file(gspo_weights_path)
    incompatible = model.load_state_dict(gspo_weights, strict=False)
    print(f"Loaded GSPO weights from {gspo_weights_path}")
    print(f"  Missing keys (fresh init): {len(incompatible.missing_keys)}")
else:
    print(f"WARNING: GSPO weights not found at {gspo_weights_path}")

if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

In [ ]:
# ============================================================
# Verification functions
# ============================================================
import re

# Try importing from stem_rewards / verify_answers
_verify_imported = False
try:
    from training.scripts.verify_answers import verify, extract_answer
    _verify_imported = True
    print("Imported verify functions from training.scripts.verify_answers")
except ImportError:
    try:
        import sys
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.verify_answers import verify, extract_answer
        _verify_imported = True
        print("Imported verify functions (via Drive path)")
    except ImportError:
        print("WARNING: verify_answers not found, using fallback")

if not _verify_imported:
    import sympy

    def extract_answer(text):
        if "</think>" in text:
            text = text.split("</think>")[-1].strip()
        boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
        if boxed:
            return boxed[-1].strip()
        numbers = re.findall(r'[-+]?\d*\.?\d+', text)
        return numbers[-1] if numbers else text.strip()

    def verify_simple(answer, truth, domain):
        if domain == "math":
            try:
                pred = sympy.sympify(answer)
                gold = sympy.sympify(truth)
                return sympy.simplify(pred - gold) == 0
            except Exception:
                return answer.strip() == str(truth).strip()
        elif domain == "physics":
            try:
                pred_num = float(re.findall(r'[-+]?\d*\.?\d+', answer)[0])
                gold_num = float(re.findall(r'[-+]?\d*\.?\d+', str(truth))[0])
                return abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05
            except (ValueError, IndexError):
                return answer.strip() == str(truth).strip()
        else:
            return answer.strip().lower() == str(truth).strip().lower()


def verify_completion(completion, problem):
    """Verify a single completion against its problem's ground truth."""
    answer = extract_answer(completion)
    truth = problem.get("ground_truth", problem.get("answer", ""))
    domain = problem.get("domain", "math")

    if _verify_imported:
        result = verify(
            answer=answer, truth=truth, domain=domain,
            question_type=problem.get("type", "calc"),
            test_cases=problem.get("test_cases"),
        )
        return result.correct
    else:
        return verify_simple(answer, truth, domain)


print("Verification functions ready")

In [ ]:
# ============================================================
# RAFT++ Training Loop with GVM-RAFT + Negative Saving
# (with Colab disconnect recovery)
# ============================================================
from trl import SFTConfig, SFTTrainer
from datasets import Dataset


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(output_dir, latest)
    print(f"  Found checkpoint: {path}")
    return path


def evaluate_accuracy(model, tokenizer, problems, sample_size=50):
    """Quick accuracy evaluation on a sample of problems."""
    FastLanguageModel.for_inference(model)
    sample = random.sample(problems, min(sample_size, len(problems)))
    correct = 0
    domain_stats = defaultdict(lambda: {"total": 0, "correct": 0})

    for problem in sample:
        domain = problem.get("domain", "math")
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=MAX_PROMPT_LENGTH
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=MAX_COMPLETION,
                temperature=0.7, do_sample=True,
            )
        completion = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        domain_stats[domain]["total"] += 1
        if verify_completion(completion, problem):
            correct += 1
            domain_stats[domain]["correct"] += 1

    FastLanguageModel.for_training(model)
    accuracy = correct / len(sample) if sample else 0
    return accuracy, dict(domain_stats)


# ---- Resume support ----
progress_path = os.path.join(OUTPUT_DIR, "progress.json")
start_round = 0
round_metrics = []
prev_accuracy = 0.0
total_negatives_saved = 0

if os.path.exists(progress_path):
    with open(progress_path) as f:
        saved = json.load(f)
    round_metrics = saved.get("round_metrics", [])
    prev_accuracy = saved.get("last_accuracy", 0)
    baseline_acc = saved.get("baseline_acc", 0)
    total_negatives_saved = saved.get("total_negatives_saved", 0)
    start_round = len(round_metrics)
    if start_round > 0:
        print(f"Resuming from round {start_round + 1} (prev accuracy: {prev_accuracy:.3f})")
        last_adapter = os.path.join(OUTPUT_DIR, f"round_{start_round}_adapter", "adapter_model.safetensors")
        if os.path.exists(last_adapter):
            from safetensors.torch import load_file as _load_rd
            model.load_state_dict(_load_rd(last_adapter), strict=False)
            print(f"  Loaded adapter from round {start_round}")

if start_round == 0:
    # Fresh start — clear negatives file and evaluate baseline
    if os.path.exists(NEGATIVES_PATH):
        os.remove(NEGATIVES_PATH)
    print("Evaluating baseline (GSPO checkpoint)...")
    baseline_acc, baseline_domains = evaluate_accuracy(model, tokenizer, verifiable_problems)
    prev_accuracy = baseline_acc
    print(f"Baseline accuracy: {baseline_acc:.3f}")
    for d, s in sorted(baseline_domains.items()):
        if s["total"] > 0:
            print(f"  {d}: {s['correct']}/{s['total']} = {100*s['correct']/s['total']:.1f}%")

# ---- Main RAFT++ loop ----
for round_idx in range(start_round, MAX_ROUNDS):
    print(f"\n{'='*60}")
    print(f"RAFT++ Round {round_idx + 1}/{MAX_ROUNDS} (GVM-RAFT dynamic allocation)")
    print(f"{'='*60}")

    # Step 1: GVM-RAFT generate and filter (returns correct + incorrect)
    correct_pairs, incorrect_pairs, gen_stats = generate_and_filter(
        model, tokenizer, verifiable_problems,
        n_completions=TOTAL_BUDGET_PER_PROBLEM,
    )
    print(f"Generated: {gen_stats['total_generated']}, Correct: {gen_stats['total_correct']}")
    print(f"Pass rate: {gen_stats['total_correct']/max(gen_stats['total_generated'],1)*100:.1f}%")

    if not correct_pairs:
        print("No correct completions found! Stopping RAFT++.")
        break

    # Step 1b: Save negatives for DPO reuse (arXiv 2505.24850)
    n_saved = save_negatives(incorrect_pairs, NEGATIVES_PATH, round_idx + 1)
    total_negatives_saved += n_saved

    # Step 2: Domain balance
    balanced_pairs = domain_balance(correct_pairs)
    print(f"After domain balancing: {len(balanced_pairs)} training examples")

    # Step 3: Format as SFT dataset
    sft_data = []
    for prompt_text, completion_text, problem in balanced_pairs:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem["prompt"]},
            {"role": "assistant", "content": completion_text},
        ]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        sft_data.append({"text": formatted})

    sft_dataset = Dataset.from_list(sft_data)
    print(f"SFT dataset: {len(sft_dataset)} examples")

    # Step 4: SFT on correct completions
    round_output = os.path.join(OUTPUT_DIR, f"round_{round_idx + 1}")
    sft_config = SFTConfig(
        output_dir=round_output,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        learning_rate=SFT_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_seq_length=SFT_MAX_SEQ_LENGTH,
        bf16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=1,
        optim="adamw_torch_fused",
        seed=42 + round_idx,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    sft_resume = find_latest_checkpoint(round_output)
    print(f"Starting SFT (round {round_idx + 1})...")
    result = trainer.train(resume_from_checkpoint=sft_resume)
    print(f"SFT loss: {result.training_loss:.4f}")

    del trainer
    torch.cuda.empty_cache()

    # Step 5: Evaluate
    round_acc, round_domains = evaluate_accuracy(model, tokenizer, verifiable_problems)
    improvement = round_acc - prev_accuracy

    print(f"\nRound {round_idx + 1} accuracy: {round_acc:.3f} (delta: {improvement:+.3f})")
    for d, s in sorted(round_domains.items()):
        if s["total"] > 0:
            print(f"  {d}: {s['correct']}/{s['total']} = {100*s['correct']/s['total']:.1f}%")

    round_metrics.append({
        "round": round_idx + 1,
        "sft_loss": result.training_loss,
        "accuracy": round_acc,
        "improvement": improvement,
        "correct_pairs": len(correct_pairs),
        "incorrect_pairs": len(incorrect_pairs),
        "balanced_pairs": len(balanced_pairs),
        "gen_stats": gen_stats,
        "domain_results": round_domains,
        "negatives_saved": n_saved,
    })

    # Save adapter + progress for Colab disconnect recovery
    round_adapter_path = os.path.join(OUTPUT_DIR, f"round_{round_idx + 1}_adapter")
    model.save_pretrained(round_adapter_path)
    tokenizer.save_pretrained(round_adapter_path)
    with open(progress_path, "w") as f:
        json.dump({
            "round_metrics": round_metrics,
            "last_accuracy": round_acc,
            "baseline_acc": baseline_acc,
            "total_negatives_saved": total_negatives_saved,
        }, f, indent=2, default=str)
    print(f"  Progress saved (round {round_idx + 1})")

    # Step 6: Early stop check
    if improvement < EARLY_STOP_THRESHOLD:
        print(f"\nEarly stopping: improvement {improvement:.4f} < threshold {EARLY_STOP_THRESHOLD}")
        break

    prev_accuracy = round_acc

print(f"\nRAFT++ complete!")
if round_metrics:
    print(f"  Final accuracy: {round_metrics[-1]['accuracy']:.3f}")
print(f"  Total negatives saved for DPO: {total_negatives_saved}")
print(f"  Negatives file: {NEGATIVES_PATH}")

In [ ]:
# ============================================================
# RAFT++ Training Loop (with Colab disconnect recovery)
# ============================================================
from trl import SFTConfig, SFTTrainer
from datasets import Dataset


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(output_dir, latest)
    print(f"  Found checkpoint: {path}")
    return path


def evaluate_accuracy(model, tokenizer, problems, sample_size=50):
    """Quick accuracy evaluation on a sample of problems."""
    FastLanguageModel.for_inference(model)
    sample = random.sample(problems, min(sample_size, len(problems)))
    correct = 0
    domain_stats = defaultdict(lambda: {"total": 0, "correct": 0})

    for problem in sample:
        domain = problem.get("domain", "math")
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=MAX_PROMPT_LENGTH
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=MAX_COMPLETION,
                temperature=0.7, do_sample=True,
            )
        completion = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        domain_stats[domain]["total"] += 1
        if verify_completion(completion, problem):
            correct += 1
            domain_stats[domain]["correct"] += 1

    FastLanguageModel.for_training(model)
    accuracy = correct / len(sample) if sample else 0
    return accuracy, dict(domain_stats)


# ---- Resume support: check for previous progress ----
progress_path = os.path.join(OUTPUT_DIR, "progress.json")
start_round = 0
round_metrics = []
prev_accuracy = 0.0

if os.path.exists(progress_path):
    with open(progress_path) as f:
        saved = json.load(f)
    round_metrics = saved.get("round_metrics", [])
    prev_accuracy = saved.get("last_accuracy", 0)
    baseline_acc = saved.get("baseline_acc", 0)
    start_round = len(round_metrics)
    if start_round > 0:
        print(f"Resuming from round {start_round + 1} (prev accuracy: {prev_accuracy:.3f})")
        # Load adapter from last completed round
        last_adapter = os.path.join(OUTPUT_DIR, f"round_{start_round}_adapter", "adapter_model.safetensors")
        if os.path.exists(last_adapter):
            from safetensors.torch import load_file as _load_rd
            model.load_state_dict(_load_rd(last_adapter), strict=False)
            print(f"  Loaded adapter from round {start_round}")

if start_round == 0:
    # Fresh start — evaluate baseline
    print("Evaluating baseline (GSPO checkpoint)...")
    baseline_acc, baseline_domains = evaluate_accuracy(model, tokenizer, verifiable_problems)
    prev_accuracy = baseline_acc
    print(f"Baseline accuracy: {baseline_acc:.3f}")
    for d, s in sorted(baseline_domains.items()):
        if s["total"] > 0:
            print(f"  {d}: {s['correct']}/{s['total']} = {100*s['correct']/s['total']:.1f}%")

# ---- Main RAFT++ loop ----
for round_idx in range(start_round, MAX_ROUNDS):
    print(f"\n{'='*60}")
    print(f"RAFT++ Round {round_idx + 1}/{MAX_ROUNDS}")
    print(f"{'='*60}")

    # Step 1: Generate and filter
    correct_pairs, gen_stats = generate_and_filter(
        model, tokenizer, verifiable_problems,
        n_completions=N_COMPLETIONS, batch_size=GEN_BATCH_SIZE,
    )
    print(f"Generated: {gen_stats['total_generated']}, Correct: {gen_stats['total_correct']}")
    print(f"Pass rate: {gen_stats['total_correct']/max(gen_stats['total_generated'],1)*100:.1f}%")

    if not correct_pairs:
        print("No correct completions found! Stopping RAFT++.")
        break

    # Step 2: Domain balance
    balanced_pairs = domain_balance(correct_pairs)
    print(f"After domain balancing: {len(balanced_pairs)} training examples")

    # Step 3: Format as SFT dataset
    sft_data = []
    for prompt_text, completion_text, problem in balanced_pairs:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem["prompt"]},
            {"role": "assistant", "content": completion_text},
        ]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        sft_data.append({"text": formatted})

    sft_dataset = Dataset.from_list(sft_data)
    print(f"SFT dataset: {len(sft_dataset)} examples")

    # Step 4: SFT on correct completions
    round_output = os.path.join(OUTPUT_DIR, f"round_{round_idx + 1}")
    sft_config = SFTConfig(
        output_dir=round_output,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        learning_rate=SFT_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_seq_length=SFT_MAX_SEQ_LENGTH,
        bf16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=1,
        optim="adamw_torch_fused",
        seed=42 + round_idx,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    # Resume SFT from checkpoint if Colab disconnected during training
    sft_resume = find_latest_checkpoint(round_output)
    print(f"Starting SFT (round {round_idx + 1})...")
    result = trainer.train(resume_from_checkpoint=sft_resume)
    print(f"SFT loss: {result.training_loss:.4f}")

    del trainer
    torch.cuda.empty_cache()

    # Step 5: Evaluate
    round_acc, round_domains = evaluate_accuracy(model, tokenizer, verifiable_problems)
    improvement = round_acc - prev_accuracy

    print(f"\nRound {round_idx + 1} accuracy: {round_acc:.3f} (delta: {improvement:+.3f})")
    for d, s in sorted(round_domains.items()):
        if s["total"] > 0:
            print(f"  {d}: {s['correct']}/{s['total']} = {100*s['correct']/s['total']:.1f}%")

    round_metrics.append({
        "round": round_idx + 1,
        "sft_loss": result.training_loss,
        "accuracy": round_acc,
        "improvement": improvement,
        "correct_pairs": len(correct_pairs),
        "balanced_pairs": len(balanced_pairs),
        "gen_stats": gen_stats,
        "domain_results": round_domains,
    })

    # Save adapter + progress for Colab disconnect recovery
    round_adapter_path = os.path.join(OUTPUT_DIR, f"round_{round_idx + 1}_adapter")
    model.save_pretrained(round_adapter_path)
    tokenizer.save_pretrained(round_adapter_path)
    with open(progress_path, "w") as f:
        json.dump({
            "round_metrics": round_metrics,
            "last_accuracy": round_acc,
            "baseline_acc": baseline_acc,
        }, f, indent=2, default=str)
    print(f"  Progress saved (round {round_idx + 1})")

    # Step 6: Early stop check
    if improvement < EARLY_STOP_THRESHOLD:
        print(f"\nEarly stopping: improvement {improvement:.4f} < threshold {EARLY_STOP_THRESHOLD}")
        break

    prev_accuracy = round_acc

print(f"\nRAFT++ complete! Final accuracy: {round_metrics[-1]['accuracy']:.3f}" if round_metrics else "No rounds completed")

In [ ]:
# ============================================================
# Save final adapter + metrics
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final RAFT++ adapter saved to {final_adapter_path}")

# Save evaluation metrics
eval_metrics = {
    "stage": "raft_plus",
    "pipeline_position": "3 of 5",
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "improvement": final_acc - baseline_acc,
    "domain_results": {
        d: {
            "accuracy": s["correct"] / s["total"] if s["total"] > 0 else 0,
            "total": s["total"],
        }
        for d, s in final_domains.items()
    },
    "rounds": round_metrics,
    "total_negatives_saved": total_negatives_saved,
    "negatives_path": NEGATIVES_PATH,
}
eval_path = os.path.join(OUTPUT_DIR, "raft_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Eval metrics saved to {eval_path}")

# Save training config
config = {
    "stage": "raft_plus",
    "pipeline": "SFT -> GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "3 of 5",
    "base_model": BASE_MODEL,
    "gspo_checkpoint": GSPO_CHECKPOINT,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "total_budget_per_problem": TOTAL_BUDGET_PER_PROBLEM,
    "max_rounds": MAX_ROUNDS,
    "actual_rounds": len(round_metrics),
    "early_stop_threshold": EARLY_STOP_THRESHOLD,
    # GVM-RAFT (arXiv 2504.11343)
    "gvm_pilot_size": GVM_PILOT_SIZE,
    "gvm_min_additional": GVM_MIN_ADDITIONAL,
    "gvm_dynamic_allocation": True,
    # Negative saving (arXiv 2505.24850)
    "negatives_saved": total_negatives_saved,
    "negatives_path": NEGATIVES_PATH,
    # SFT
    "sft_lr": SFT_LR,
    "sft_epochs": SFT_EPOCHS,
    "lora_r": effective_r,
    "lora_alpha": gspo_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems),
    "domain_balancing": True,
    "references": [
        "GVM-RAFT arXiv:2504.11343",
        "Negative Reuse arXiv:2505.24850",
        "ReST arXiv:2308.08998",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = False
HF_REPO_ID = "Siesher/mits-qwen3-4b-raft"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! RAFT++ adapter ready for AdaSTaR (next stage).")
print(f"Negatives saved at {NEGATIVES_PATH} for DPO reuse.")

In [ ]:
# ============================================================
# Save final adapter + metrics
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final RAFT++ adapter saved to {final_adapter_path}")

# Save evaluation metrics
eval_metrics = {
    "stage": "raft_plus",
    "pipeline_position": "3 of 5",
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "improvement": final_acc - baseline_acc,
    "domain_results": {
        d: {
            "accuracy": s["correct"] / s["total"] if s["total"] > 0 else 0,
            "total": s["total"],
        }
        for d, s in final_domains.items()
    },
    "rounds": round_metrics,
}
eval_path = os.path.join(OUTPUT_DIR, "raft_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Eval metrics saved to {eval_path}")

# Save training config
config = {
    "stage": "raft_plus",
    "pipeline": "SFT -> GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "3 of 5",
    "base_model": BASE_MODEL,
    "gspo_checkpoint": GSPO_CHECKPOINT,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "n_completions": N_COMPLETIONS,
    "max_rounds": MAX_ROUNDS,
    "actual_rounds": len(round_metrics),
    "early_stop_threshold": EARLY_STOP_THRESHOLD,
    "sft_lr": SFT_LR,
    "sft_epochs": SFT_EPOCHS,
    "lora_r": effective_r,
    "lora_alpha": gspo_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems),
    "domain_balancing": True,
    "references": [
        "RAFT arXiv:2504.11343",
        "ReST arXiv:2308.08998",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = False
HF_REPO_ID = "Siesher/mits-qwen3-4b-raft"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! RAFT++ adapter ready for AdaSTaR (next stage).")